**Connect Scripts**

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from Scripts import FileHandler as fh

**Inspect Data Set**

In [ ]:
# load data set
import kagglehub
from pathlib import Path

downloadPath = kagglehub.dataset_download('yasserh/titanic-dataset')
dataPath = Path(downloadPath)

print(f'Content of {dataPath}:')
for item in dataPath.iterdir():
    print(f"    -{item.name} ({'Folder' if item.is_dir() else 'File'})")


**Inspect Data Set**

In [ ]:
# Inspect data set
import pandas as pd

dfTitanic = pd.read_csv(f'{downloadPath}/Titanic-Dataset.csv')

# print first 5 lines
display(dfTitanic.head())

# check data set shape
print(f"\n Titanic Dataset shape : {dfTitanic.shape}")

# check available data types
print("\n Data Type Count")
print(dfTitanic.dtypes.value_counts())

# check missing values in data set
print("\n Missing values in Data set")
dfTitanic.info()

# statistical summary
print("\n statistical summary")
display(dfTitanic.describe())

# check distribution
print("\n Data set distribution")
print(dfTitanic['Survived'].value_counts())
print(dfTitanic['Survived'].value_counts(normalize=True)* 100)


**Train/Validate/Test Split**
- avoid cross contamination (don't need to get influence by Training data)
- 70% : Train ; 15% : Validation ; 15% : Test

In [ ]:
from sklearn.model_selection import train_test_split

dfTrain, dfTemp = train_test_split(dfTitanic, test_size=0.30, random_state=42)
dfVal, dfTest = train_test_split(dfTemp, test_size=0.5, random_state=42)

print(f"Dataset Size: {len(dfTitanic)} | Train Size: {len(dfTrain)} | Validate Size: {len(dfVal)} | Test Size: {len(dfTest)}")
fh.saveDataSets(dfTrain, dfVal, dfTest, "titanicdataset/preprocessing", "01_datasplit")

**Handle Missing Values**
- Drop the columns
- Imputation/ Replace
    - mean
    - meadian
    - frequent
    - null value

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("titanicdataset/preprocessing", "01_datasplit")

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
# drop columns
dropList = ['Cabin']

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfVal.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
# replace with median value
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
medianReplaceList = ['Age']
print("missing in Train: ", dfTrain['Age'].isnull().sum())
print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
for column in medianReplaceList:
    if column in dfTrain.columns:
        imputer.fit(dfTrain[[column]])
        print(f"Imputed(median) value: {imputer.statistics_}")

        dfTrain[[column]] = imputer.transform(dfTrain[[column]])

        if column in dfVal:
            dfVal[[column]] = imputer.transform(dfVal[[column]])

        if column in dfTest:
            dfTest[[column]] = imputer.transform(dfTest[[column]])
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
print("missing in Train: ", dfTrain['Age'].isnull().sum())

In [ ]:
# replace with frequent value

frequentReplaceList = ['Embarked']

print("missing in Train: ", dfTrain['Embarked'].isnull().sum())
for column in frequentReplaceList:
    if column in dfTrain.columns:
        modeValue = dfTrain[column].mode()[0]
        print(f"Imputed(frequent) value: {modeValue}")

        dfTrain[column] = dfTrain[column].fillna(modeValue)

        if column in dfVal:
            dfVal[column] = dfVal[column].fillna(modeValue)

        if column in dfTest:
            dfTest[column] = dfTest[column].fillna(modeValue)

print("missing in Train: ", dfTrain['Embarked'].isnull().sum())

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "titanicdataset/preprocessing", "02_handlemissingvalues")

**Detect Outliers**
- Not Need to handle:
    - predefined ranges
    - Years
    - Months
    - Timestamps
- Required to handle:
    - calculated
    - physical
- May be:
    - counts
    - Targets  
      
        
- Outlier finding methods:
    - Q1:first quartile - 25% of data falls below this value
    - Q2:second quartile - split data in half
    - Q3:third quartile - 75% of data falls below this value
    - IQR = Q3 -Q1:interquartile value: middle 50% of the spread
    - Oulier boundary: 
        - lower fence: Q1 - 1.5*IQR
        - upper fence: Q3 + 1.5*IQR

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("titanicdataset/preprocessing", "02_handlemissingvalues")

In [ ]:
numCol = dfTrain.select_dtypes(exclude=["str"]).columns
print(numCol)

In [ ]:
outlierCheckList = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
validColmns = [col for col in outlierCheckList if col in dfTrain.columns]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nCols = 3
nRows = int(np.ceil(len(validColmns)/nCols))

fig, axes = plt.subplots(nRows, nCols, figsize=(15, 3.5 * nRows))
axes = axes.flatten()

for idx, col in enumerate(validColmns):
    ax = axes[idx]
    ax.plot(dfTrain.index, dfTrain[col], marker='o', linestyle='', alpha=0.5, markersize=4)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

for idx in range(len(validColmns), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Outlier check: scatterplots by column', fontsize=14, y=1.01)
plt.show()

In [ ]:
for col in outlierCheckList:
    if col not in dfTrain.columns:
        continue

    Q1 = dfTrain[col].quantile(0.25)
    Q3 = dfTrain[col].quantile(0.75)
    IQR = Q3-Q1

    lowerFence = Q1 - 1.5 * IQR
    upperFence = Q3 + 1.5 * IQR

    outlierRows = dfTrain[(dfTrain[col] < lowerFence) | (dfTrain[col] > upperFence)]
    outlierValues = outlierRows[col].sort_values()

    print(f"\ncolumn: {col}")
    print(f"Q1={Q1:.2f} | Q3={Q3:.2f} | IQR={IQR:.2f}")
    print(f"Fences: [{lowerFence:.2f}, {upperFence:.2f}]")
    print(f"Outlier Count: {len(outlierValues)} | Percentage: {(len(outlierValues)/ len(dfTrain)) * 100}")

    if len(outlierValues) > 0:
        lowOutliers = outlierValues[outlierValues < lowerFence]
        highOutliers = outlierValues[outlierValues > upperFence]

        if len(lowOutliers) > 0:
            print(f"    below lower fence: {list(lowOutliers)}")

        if len(highOutliers) > 0:
            print(f"    above upper fence: {list(highOutliers)}")
    else:
        print("no outliers found")

In [ ]:
# set manual bounds
# use domain knowledge

investigateUpper = {
    'SibSp' : 5,
    'Parch' : 4,
    'Fare' : 500
}

investigateLower = {
    'Age' : 3
}

In [ ]:
for col, val in investigateUpper.items():
    rows = dfTrain[dfTrain[col] > val]
    print(f"{col} value:{val} : {len(rows)} rows")
    print(rows.to_string())

In [ ]:
rows = dfTrain[(dfTrain['Pclass']==1) & (dfTrain['Embarked']=='C')]
print(rows.to_string())

In [ ]:
for col, val in investigateLower.items():
    rows = dfTrain[dfTrain[col] < val]
    print(f"{col} value:{val} : {len(rows)} rows")
    print(rows.to_string())

In [ ]:
# define which columns need capping
capList = ['Fare']

caps = {}

for colmn in capList:
    if colmn in dfTrain.columns:
        caps[colmn] = dfTrain[colmn].quantile(0.99) # use 99% from train set
        print(f"{colmn}: cap = {caps[colmn]}")

In [ ]:
def applyOutlierTreatment(df, caps, dataset_name:str):
    df = df.copy()

    for colmn, cap in caps.items():
        if colmn in df.columns:
            before = (df[colmn] > cap).sum()
            df[colmn] = df[col].clip(upper=cap)
            if before > 0:
                print(f"{dataset_name}: capped {before} values in {colmn} at {cap}")

    return df

dfTrain = applyOutlierTreatment(dfTrain, caps, "Train")
# dfVal = applyOutlierTreatment(dfVal, caps, "Val")
# dfTest = applyOutlierTreatment(dfTest, caps, "Test")

In [ ]:
rows = dfTrain[(dfTrain['Fare'] > 256)]
print(rows.to_string())

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "titanicdataset/preprocessing", "03_handleoutliers")